# 29 · External test set CSAW-S (test images, three radiologists) — CPU, ~10 min
Reads `data/external/CSAW-S.zip` (Zenodo 10.5281/zenodo.4030660, restricted access granted to the user; CC BY-NC-ND 4.0, academic use).
Writes `data/external/csaws_mass_256_jit_a1/` (images, consensus masks, per-radiologist masks, metadata). Nothing is redistributed.
Rules (prespecified): consensus = at least 2 of 3 radiologists; lesion = connected component (8-connectivity) of the consensus mask
with area >= 0.05% of the image (amendment 1, recorded below before any evaluation; the prespecified 0.5% subset is flagged and also reported); crops exactly as the study (scale 1.5-3.0 x lesion box, shift 0.8, seed from MD5 of the lesion id,
256 x 256); image intensities with the study's 1st-99.5th percentile window.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, glob, re, json, zipfile, hashlib
import numpy as np, pandas as pd, cv2, time
PROJECT = '/content/drive/MyDrive/quantum-placement'
ZIP = f'{PROJECT}/data/external/CSAW-S.zip'; OUT = f'{PROJECT}/data/external/csaws_mass_256_jit_a1'
LOCAL = '/content/csaws'
CROP_SIZE, SCALE_LO, SCALE_HI, SHIFT, MIN_FRAC = 256, 1.5, 3.0, 0.8, 0.0005
PRESPEC_FRAC = 0.005          # rule of the stored protocol; lesions meeting it are flagged (meets_prespecified_0p5pct)
md5 = hashlib.md5()
with open(ZIP, 'rb') as f:
    for ch in iter(lambda: f.read(1 << 24), b''): md5.update(ch)
print('md5', md5.hexdigest(), '| expected 3182631e19de57bb0cbf5f22905e1133')
assert md5.hexdigest() == '3182631e19de57bb0cbf5f22905e1133'
if not os.path.isdir(LOCAL): zipfile.ZipFile(ZIP).extractall(LOCAL)
TEST = [d for d in glob.glob(f'{LOCAL}/**/test_data', recursive=True)][0]
ANN = sorted(d for d in os.listdir(TEST) if d.startswith('annotator'))
print('test_data:', sorted(os.listdir(TEST)), '| annotators:', ANN)
for d in ('images', 'masks') + tuple(f'masks_{a}' for a in ANN): os.makedirs(f'{OUT}/{d}', exist_ok=True)
# protocol amendment 1 (stored once, before any model is evaluated on CSAW-S or CDD-CESM and before any revision-5 run is trained)
AMEND = f'{PROJECT}/results/p2_rev5/PROTOCOL_rev5_amendment1.txt'; os.makedirs(os.path.dirname(AMEND), exist_ok=True)
AMEND_TXT = """Paper 2 - revision 5, protocol amendment 1 (CSAW-S lesion size rule).
Recorded before any model was evaluated on CSAW-S or CDD-CESM and before any revision-5 run was trained.
Reason: with the stored rule (one lesion = one 8-connected component of the consensus mask with area >= 0.5% of the image) the
CSAW-S test set gave 4 lesions in 4 patients (26 test images; 17 smaller consensus components excluded), too few for the planned
cluster bootstrap and Holm adjustment. Only the lesion inventory (counts and sizes) was seen; no prediction or Dice on CSAW-S exists.
Change: the minimum component area is 0.05% of the image (smaller components are outline fragments). Lesions that also meet the
stored 0.5% rule are flagged (meets_prespecified_0p5pct) and the contrasts are also reported for that subset. Everything else in the
stored protocol is unchanged (consensus of at least two of three radiologists, crops, window, references, analysis).
Output folder: data/external/csaws_mass_256_jit_a1 (the folder of the first preparation, csaws_mass_256_jit, is not used)."""
if not os.path.exists(AMEND):
    open(AMEND, 'w').write(AMEND_TXT + '\nwritten (UTC): ' + time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()) + '\n')
assert open(AMEND).read().startswith(AMEND_TXT)
print(open(AMEND).read().splitlines()[-1])

In [ ]:
# helpers identical to notebook 25 (windowing and randomized crop)
def to_uint8(img):
    img = img.astype(np.float32); nz = img[img > 0]
    lo, hi = (np.percentile(nz, 1), np.percentile(nz, 99.5)) if nz.size else (img.min(), img.max())
    return (np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)
def crop_window(mask, rid, size=CROP_SIZE):
    ys, xs = np.where(mask > 0)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    L = max(y1 - y0, x1 - x0)
    rng = np.random.default_rng(int(hashlib.md5(rid.encode()).hexdigest()[:8], 16))
    scale = rng.uniform(SCALE_LO, SCALE_HI); side = max(int(L * scale), 64)
    slack_y, slack_x = (side - (y1 - y0)) / 2, (side - (x1 - x0)) / 2
    dy, dx = rng.uniform(-SHIFT, SHIFT, 2)
    cy = int(round((y0 + y1) / 2 + dy * slack_y)); cx = int(round((x0 + x1) / 2 + dx * slack_x))
    return cy - side // 2, cx - side // 2, side, dict(bbox_h=int(y1 - y0), bbox_w=int(x1 - x0), crop_side=side,
                                                      scale=round(float(scale), 4), shift_y=round(float(dy), 4), shift_x=round(float(dx), 4))
def cut(a, top, left, side, interp):
    H, W = a.shape; pad = max(0, -top, -left, top + side - H, left + side - W)
    if pad: a = np.pad(a, pad, mode='constant'); top += pad; left += pad
    return cv2.resize(a[top:top + side, left:left + side], (CROP_SIZE, CROP_SIZE), interpolation=interp), bool(pad)

In [ ]:
# images and radiologist cancer outlines
imgs = {}
for p in glob.glob(f'{TEST}/anonymized_dataset/**/*.png', recursive=True):
    b = os.path.basename(p)[:-4]
    if re.search(r'_(cancer|calcifications|axillary_lymph_nodes|thick_vessels|foreign_object|skin|nipple|text|non-mammary_tissue|pectoral_muscle|mammary_gland)$', b): continue
    sid = b.replace('_8bit', '')
    if b.endswith('_8bit') or sid not in imgs: imgs[sid] = p
rows, n_small, comp_fracs = [], 0, []
for sid, ip in sorted(imgs.items()):
    g = cv2.imread(ip, cv2.IMREAD_UNCHANGED)
    if g.ndim == 3: g = cv2.cvtColor(g, cv2.COLOR_BGR2GRAY)
    H, W = g.shape; g8 = to_uint8(g)
    ann = {}
    for a in ANN:
        hits = glob.glob(f'{TEST}/{a}/**/{sid}_cancer.png', recursive=True)
        m = (cv2.imread(hits[0], 0) > 0) if hits else np.zeros((H, W), bool)
        if m.shape != (H, W): m = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST) > 0
        ann[a] = m
    cons = (np.sum([ann[a] for a in ANN], 0) >= 2).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(cons, connectivity=8)
    comp_fracs += [st[k, cv2.CC_STAT_AREA] / (H * W) for k in range(1, n)]
    comps = sorted([k for k in range(1, n) if st[k, cv2.CC_STAT_AREA] >= MIN_FRAC * H * W], key=lambda k: (st[k, cv2.CC_STAT_TOP], st[k, cv2.CC_STAT_LEFT]))
    n_small += sum(1 for k in range(1, n) if st[k, cv2.CC_STAT_AREA] < MIN_FRAC * H * W)
    for j, k in enumerate(comps):
        rid = f'csaws_{re.sub(r"[^A-Za-z0-9]", "", sid)}_c{j}'
        lesion = (lab == k).astype(np.uint8)
        top, left, side, info = crop_window(lesion, rid)
        ic, padded = cut(g8, top, left, side, cv2.INTER_AREA)
        mc, _ = cut(lesion, top, left, side, cv2.INTER_NEAREST)
        cv2.imwrite(f'{OUT}/images/{rid}.png', ic); cv2.imwrite(f'{OUT}/masks/{rid}.png', (mc > 0).astype(np.uint8) * 255)
        for a in ANN:
            ma, _ = cut(ann[a].astype(np.uint8), top, left, side, cv2.INTER_NEAREST)
            cv2.imwrite(f'{OUT}/masks_{a}/{rid}.png', (ma > 0).astype(np.uint8) * 255)
        rows.append(dict(roi_id=rid, image=sid, patient=sid.split('_')[0], padded=padded, mask_frac=round(float((mc > 0).mean()), 4),
                         lesion_frac_of_image=round(float(lesion.mean()), 5),
                         meets_prespecified_0p5pct=bool(lesion.mean() >= PRESPEC_FRAC), **info))
meta = pd.DataFrame(rows); meta.to_csv(f'{OUT}/metadata.csv', index=False)
print('test images:', len(imgs), '| lesions:', len(meta), '| patients:', meta.patient.nunique(), '| images without a lesion:',
      len(imgs) - meta.image.nunique(), '| components below 0.05% excluded:', n_small)
print('lesions meeting the stored 0.5% rule:', int(meta.meets_prespecified_0p5pct.sum()))
print('all consensus components, area as fraction of the image:', pd.Series(comp_fracs).describe(percentiles=[.1, .25, .5, .75, .9]).round(6).to_dict())
print(meta[['scale', 'mask_frac', 'lesion_frac_of_image']].describe().round(4).to_string())

In [ ]:
# overlay sheet (green = consensus, per-radiologist outlines in red / yellow / blue)
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
ids = meta.roi_id.tolist(); cols = 8; rws = int(np.ceil(len(ids) / cols))
fig, ax = plt.subplots(rws, cols, figsize=(16, 2.2 * rws), squeeze=False)
colours = [(255, 60, 60), (255, 220, 0), (60, 140, 255)]
for a_, r in zip(ax.ravel(), ids):
    im = cv2.cvtColor(cv2.imread(f'{OUT}/images/{r}.png', 0), cv2.COLOR_GRAY2RGB)
    for a, c in zip(ANN, colours):
        cs, _ = cv2.findContours((cv2.imread(f'{OUT}/masks_{a}/{r}.png', 0) > 127).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE); cv2.drawContours(im, cs, -1, c, 1)
    cs, _ = cv2.findContours((cv2.imread(f'{OUT}/masks/{r}.png', 0) > 127).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE); cv2.drawContours(im, cs, -1, (60, 220, 60), 2)
    a_.imshow(im); a_.set_title(r[-14:], fontsize=6)
for a_ in ax.ravel(): a_.axis('off')
plt.tight_layout(); plt.savefig(f'{OUT}/sheet.png', dpi=80); print('DONE')